In [1]:
import pandas as pd

In [2]:
from etna.datasets.tsdataset import TSDataset
from etna.transforms import (
    LogTransform,
    StandardScalerTransform,
    LinearTrendTransform,
    SegmentEncoderTransform,
    LagTransform,
    DateFlagsTransform,
    MeanTransform,
    MinMaxDifferenceTransform,
)
from etna.metrics import SMAPE, WAPE
from etna.models import CatBoostMultiSegmentModel, SeasonalMovingAverageModel
from etna.pipeline import Pipeline
from etna.analysis import plot_forecast

In [3]:
def train_and_evaluate_model(
    ts,
    model,
    transforms,
    horizon,
    metrics,
    print_metrics=False,
    print_plots=False,
    n_train_samples=None
):
    """
    Обучает модель, вычисляет прогнозы для
    тестовой выборки и строит график прогнозов.
    Параметры
    ----------
    ts: pandas.DataFrame
    Временной ряд.
    model: instance of class etna
    Экземпляр класса библиотеки etna.
    transforms: list
    Список преобразований.
    horizon: int
    Горизонт прогнозирования.
    metrics: instance of class etna.metrics.SMAPE/
    MAE/R2/MAPE/MedAE/MSLE
    Метрика качества.
    print_metrics: bool, по умолчанию False
    Печать метрик.
    print_plots: bool, по умолчанию False
    Печать графиков прогнозов.
    n_train_sample: int
    n последних наблюдений обучающей выборки
    на графике прогнозов.
    """
    if not print_plots and n_train_samples is not None:
        raise ValueError(
            "Параметр n_train_samples задается при print_plots=True"
        )
    # разбиваем набор на обучающую и тестовую выборки
    # с учетом временной структуры, размер тестовой
    # выборки задаем равным горизонту
    train_ts, test_ts = ts.train_test_split(test_size=horizon)
    # создаем конвейер
    pipe = Pipeline(
        model=model,
        transforms=transforms,
        horizon=horizon)
    
    # обучаем конвейер
    pipe.fit(train_ts)
    # получаем прогнозы
    forecast_ts = pipe.forecast()
    # оцениваем качество прогнозов по сегментам
    segment_metrics = metrics(test_ts, forecast_ts)
    segment_metrics = pd.Series(segment_metrics)
    
    if print_metrics:
        # print(segment_metrics.to_string())
        # print("")
        # оцениваем качество прогнозов в среднем
        print(f"WAPE:" f"{sum(segment_metrics) / len(segment_metrics)}")
    
    if print_plots:
        # визуализируем прогнозы, здесь n_train_samples
        # – n последних наблюдений в обучающей выборке
        plot_forecast(forecast_ts, test_ts,
        train_ts, n_train_samples=n_train_samples)

    return forecast_ts

In [4]:
from demand_forecast.src.env import EXPERIMENTS_DIR

In [5]:
EXPERIMENTS_DIR

PosixPath('/Users/sergey.rubtsovenko/Projects/research/demand_forecast/experiments')

# Data

In [6]:
trend_tours_df = pd.read_parquet(EXPERIMENTS_DIR / 'experiment_1/trend_tours')
trend_tours_df.head()

,tour_id,avg_yoy_growth,weighted_yoy_growth,tickets_2026,tickets_2025
0,480657,511.976454,12.167730,14445,1097
1,5089,5.417738,5.842763,10401,1520
2,713703,54.023506,5.678423,12876,1928
3,301483,10.560220,5.154801,7435,1208
4,983812,4.901664,4.901664,7802,1322


In [7]:
tour_date_ts_df = pd.read_parquet(EXPERIMENTS_DIR / 'experiment_1/tour_day_sales')
tour_date_ts_df.head()

,tour_id,tour_date,tickets
0,980570,2026-03-27,22
1,194990,2026-04-01,17
2,695420,2026-03-28,9
3,798664,2026-04-14,13
4,138878,2026-03-23,18


In [12]:
tour_date_ts_df.shape

(16241513, 3)

In [8]:
analytics_forecast_df = pd.read_parquet(EXPERIMENTS_DIR / 'experiment_1/analytics_baseline')
analytics_forecast_df['tour_date'] = analytics_forecast_df['tour_date'].astype(str)
analytics_forecast_df.head()

,forecast_created_date,tour_date,tour_id,forecast,tickets
0,2026-01-26,2026-04-04,684,5.0,8
1,2026-01-26,2026-04-04,3639,4.0,0
2,2026-01-26,2026-04-04,3685,8.0,1
3,2026-01-26,2026-04-04,4325,14.0,25
4,2026-01-26,2026-04-04,4855,0.0,0


In [10]:
print('min date:', analytics_forecast_df['tour_date'].min())
print('max date:', analytics_forecast_df['tour_date'].max())

min date: 2026-02-01
max date: 2026-04-30


In [11]:
analytics_forecast_df.shape

(8891253, 5)

In [38]:
def wape(actual, forecast):
    return (actual - forecast).abs().sum() / actual.sum()

In [40]:
wape(
    actual=analytics_forecast_df['tickets'],
    forecast=analytics_forecast_df['forecast'],
)

0.9152379481683738

In [20]:
tmp_df = analytics_forecast_df.merge(tour_date_ts_df, on=['tour_id', 'tour_date'], how='left')

In [21]:
tmp_df.shape

(8891253, 6)

In [27]:
(tmp_df[tmp_df['tickets_y'].isna()]['tickets_x'] == 0)

1          True
4          True
6          True
9          True
11         True
           ... 
8891245    True
8891246    True
8891248    True
8891249    True
8891252    True
Name: tickets_x, Length: 7443069, dtype: bool

In [32]:
tmp2_df = tmp_df[tmp_df['tickets_y'].isna()]

In [36]:
sum(tmp2_df['tickets_x'] != 0) / len(tmp_df)

0.0005909178380145071

In [37]:
tmp2_df[  len(tmp2_df)

SyntaxError: incomplete input (1562925278.py, line 1)

In [41]:
tmp_df = analytics_forecast_df.merge(tour_date_ts_df, on=['tour_id', 'tour_date'], how='inner')

In [42]:
tmp_df.shape

(1448184, 6)

In [19]:
1448184 / 8891253

0.16287738072462904

In [45]:
wape(
    actual=tmp_df['tickets_x'],
    forecast=tmp_df['forecast'],
)

0.7296937377593479

# Use only trend tours

In [7]:
tour_date_ts_df = tour_date_ts_df.merge(trend_tours_df[['tour_id']], on='tour_id', how='inner')
analytics_forecast_df = analytics_forecast_df.merge(trend_tours_df[['tour_id']], on='tour_id', how='inner')

In [8]:
tour_date_ts_df = tour_date_ts_df.merge(
    analytics_forecast_df[['tour_id']].drop_duplicates(),
    on='tour_id', 
    how='inner',
)

# Modelling

In [9]:
tour_date_ts_df = tour_date_ts_df[tour_date_ts_df['tour_date'] < '2026-05-01'].reset_index(drop=True)

In [10]:
import numpy as np

In [12]:
rng = np.random.default_rng(seed=42)
tour_ids = np.random.choice(tour_date_ts_df['tour_id'].unique(), size=100, replace=False)

In [13]:
hist_sales = tour_date_ts_df

hist_sales = hist_sales[hist_sales['tour_id'].isin(tour_ids)]

hist_sales['date'] = pd.to_datetime(hist_sales['tour_date'])
hist_sales['tickets_sold'] = hist_sales['tickets']
hist_sales = hist_sales.set_index("date")

/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/3196907311.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hist_sales['date'] = pd.to_datetime(hist_sales['tour_date'])
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/3196907311.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hist_sales['tickets_sold'] = hist_sales['tickets']


In [14]:
# Fill missing dates per tour_id with linear interpolation
filled = []
for tid in hist_sales['tour_id'].unique():
    s = hist_sales[hist_sales['tour_id'] == tid][['tickets_sold']].sort_index()
    full_range = pd.date_range(s.index.min(), s.index.max(), freq='D')
    s = s.reindex(full_range)
    s['tickets_sold'] = s['tickets_sold'].interpolate(method='linear')
    s['tour_id'] = tid
    filled.append(s)

hist_sales = pd.concat(filled)
hist_sales.index.name = 'date'
print(hist_sales.shape)

(102392, 2)


In [15]:
hist_sales['timestamp'] = hist_sales.index
hist_sales = hist_sales.reset_index(drop=True)

In [16]:
hist_sales_melt = hist_sales.rename(columns={'tickets_sold': 'target', 'tour_id': 'segment'})

In [17]:
# создаем переменную – квартал, начало и конец квартала
hist_sales_melt['quarter'] = hist_sales_melt['timestamp'].dt.quarter
hist_sales_melt['quarter_start'] = hist_sales_melt['timestamp'].dt.is_quarter_start
hist_sales_melt['quarter_end'] = hist_sales_melt['timestamp'].dt.is_quarter_end
# создаем переменную – название месяца
hist_sales_melt['month_name'] = hist_sales_melt['timestamp'].dt.strftime('%b')
hist_sales_melt['month_name'] = hist_sales_melt['month_name'].astype('category')
hist_sales_melt.head()

,target,segment,timestamp,quarter,quarter_start,quarter_end,month_name
0,15.000000,518384,2023-10-05,4,False,False,Oct
1,14.857143,518384,2023-10-06,4,False,False,Oct
2,14.714286,518384,2023-10-07,4,False,False,Oct
3,14.571429,518384,2023-10-08,4,False,False,Oct
4,14.428571,518384,2023-10-09,4,False,False,Oct


In [18]:
# переводим набор данных о продажах в формат ETNA
df_ts_format = TSDataset.to_dataset(hist_sales_melt)

In [19]:
ts = TSDataset(
    df_ts_format, 
    freq='D',
    # df_exog=df_regressors_ts_format, 
    known_future='all',
)

In [20]:
ts.describe()

,start_timestamp,end_timestamp,length,num_missing,num_segments,num_exogs,num_regressors,num_known_future,freq
segments,,,,,,,,,
108124,2023-01-01,2026-04-30,1216,0,100,4,0,0,D
110581,2023-01-01,2026-04-30,1216,0,100,4,0,0,D
118750,2023-01-02,2026-04-30,1215,0,100,4,0,0,D
128852,2023-01-02,2026-04-30,1215,0,100,4,0,0,D
129952,2023-01-01,2026-04-30,1216,0,100,4,0,0,D
...,...,...,...,...,...,...,...,...,...
883385,2024-12-14,2026-04-30,503,0,100,4,0,0,D
888939,2024-12-24,2026-04-30,493,0,100,4,0,0,D
91851,2023-01-01,2026-04-30,1216,0,100,4,0,0,D


In [22]:
summary_table = ts.describe()

In [23]:
clean_tour_ids = summary_table[summary_table['num_missing'] == 0].index

In [24]:
clean_tour_ids = [int(tour_id) for tour_id in list(clean_tour_ids)]
print(len(clean_tour_ids))

93


In [25]:
hist_sales = hist_sales[hist_sales['tour_id'].isin(clean_tour_ids)]

In [26]:
hist_sales_melt = hist_sales.rename(columns={'tickets_sold': 'target', 'tour_id': 'segment'})
# создаем переменную – квартал, начало и конец квартала
hist_sales_melt['quarter'] = hist_sales_melt['timestamp'].dt.quarter
hist_sales_melt['quarter_start'] = hist_sales_melt['timestamp'].dt.is_quarter_start
hist_sales_melt['quarter_end'] = hist_sales_melt['timestamp'].dt.is_quarter_end
# создаем переменную – название месяца
hist_sales_melt['month_name'] = hist_sales_melt['timestamp'].dt.strftime('%b')
hist_sales_melt['month_name'] = hist_sales_melt['month_name'].astype('category')
hist_sales_melt.head()
# переводим набор данных о продажах в формат ETNA
df_ts_format = TSDataset.to_dataset(hist_sales_melt)
ts = TSDataset(
    df_ts_format, 
    freq='D',
    # df_exog=df_regressors_ts_format, 
    known_future='all',
)

In [27]:
summary_table = ts.describe()
len(summary_table[summary_table['num_missing'] != 0])

0

In [28]:
log = LogTransform(in_column='target')
scaler = StandardScalerTransform(in_column='target')
detrend = LinearTrendTransform(in_column='target')
seg = SegmentEncoderTransform()
lags = LagTransform(
    in_column='target',
    lags=list(range(90, 361, 30)),
    out_column='lag'
)
lags_2 = LagTransform(
    in_column='target',
    lags=[364, 365],
    out_column='lag'
)
d_flags = DateFlagsTransform(day_number_in_year=True,
    day_number_in_week=True,
    day_number_in_month=True,
    week_number_in_month=True,
    week_number_in_year=True,
    month_number_in_year=True,
    season_number=True,
    is_weekend=True,
    out_column='datetime'
)
mean90 = MeanTransform(in_column='target', window=90,
    out_column='mean90')
mean180 = MeanTransform(in_column='target', window=180,
    out_column='mean180')
mean270 = MeanTransform(in_column='target', window=270,
    out_column='mean270')

minmax270 = MinMaxDifferenceTransform(
    in_column='target', 
    window=270,
    out_column='minmax270',
)

In [29]:
# задаем горизонт прогнозирования
HORIZON = 90

In [30]:
wape = WAPE()

In [31]:
ctbst_model = CatBoostMultiSegmentModel(
    iterations=500,
    learning_rate=0.25,
    depth=5,
)

In [54]:
from copy import deepcopy
from typing import Optional

import numpy as np
import pandas as pd
from etna.transforms import ReversiblePerSegmentWrapper
from etna.transforms.base import OneSegmentTransform


class _OneSegmentLogTrendTransform(OneSegmentTransform):
    """Fits a·log(t) + b trend per segment and removes it."""

    def __init__(self, in_column: str = "target"):
        self.in_column = in_column
        self._a: Optional[float] = None
        self._b: Optional[float] = None
        self._t0: Optional[float] = None

    def _get_t(self, df: pd.DataFrame) -> np.ndarray:
        idx = df.index
        if pd.api.types.is_integer_dtype(idx):
            t = idx.astype(float).values
        else:
            t = np.array([ts.timestamp() for ts in idx])
        return t

    def fit(self, df: pd.DataFrame) -> "_OneSegmentLogTrendTransform":
        col = df[self.in_column].dropna()
        t = self._get_t(df.loc[col.index])
        self._t0 = t[0]
        log_t = np.log1p(t - self._t0)  # log(1 + Δt), safe for t==t0
        # OLS: [log_t, 1] @ [a, b]
        X = np.column_stack([log_t, np.ones_like(log_t)])
        coeffs, *_ = np.linalg.lstsq(X, col.values, rcond=None)
        self._a, self._b = coeffs
        return self

    def _predict_trend(self, df: pd.DataFrame) -> np.ndarray:
        t = self._get_t(df)
        log_t = np.log1p(t - self._t0)
        return self._a * log_t + self._b

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        trend = self._predict_trend(df)
        df[self.in_column] = df[self.in_column] - trend
        return df

    def inverse_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        trend = self._predict_trend(df)
        df[self.in_column] = df[self.in_column] + trend
        return df

    def get_regressors_info(self) -> list[str]:
        return []


class LogTrendTransform(ReversiblePerSegmentWrapper):
    """Removes a logarithmic trend (a·log(t) + b) from each segment."""

    def __init__(self, in_column: str = "target"):
        self.in_column = in_column
        super().__init__(
            transform=_OneSegmentLogTrendTransform(in_column=in_column),
            required_features=[in_column],
        )

    def get_regressors_info(self) -> list[str]:
        return []


In [55]:
logtrend = LogTrendTransform(in_column="target")

In [60]:
preprocess = [
    log, scaler, logtrend, seg, lags, lags_2, d_flags,
    mean90, mean180, mean270, minmax270
]

In [61]:
forecast_ctbst = train_and_evaluate_model(
    ts=ts,
    model=ctbst_model,
    transforms=preprocess,
    horizon=HORIZON,
    metrics=wape,
    print_metrics=True,
    # print_plots=True,
    # n_train_samples=180
)

/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0)
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0)
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0)
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0)
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0)
/var/folders/17/fpk03l2j4734d5n7zr5fllf40000gp/T/ipykernel_67613/1870950112.py:40: RuntimeWarning: invalid value encountered in log1p
  log_t = np.log1p(t - self._t0

WAPE:0.5248597043684802


In [47]:
preprocess = [
    detrend, log, scaler, seg, lags, lags_2, d_flags,
    mean90, mean180, mean270, minmax270
]

In [48]:
forecast_ctbst = train_and_evaluate_model(
    ts=ts,
    model=ctbst_model,
    transforms=preprocess,
    horizon=HORIZON,
    metrics=wape,
    print_metrics=True,
    # print_plots=True,
    # n_train_samples=180
)

/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/transforms/decomposition/detrend.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result[self.in_column] = no_trend_timeseries
/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/transforms/decomposition/detrend.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result[self.in_column] = no_trend_timeseries
/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etn

ValueError: LogPreprocess can be applied only to non-negative series

In [43]:
preprocess = [
    log, scaler, detrend, seg, lags, lags_2, d_flags,
    mean90, mean180, mean270, minmax270
]

In [45]:
forecast_ctbst = train_and_evaluate_model(
    ts=ts,
    model=ctbst_model,
    transforms=preprocess,
    horizon=HORIZON,
    metrics=wape,
    print_metrics=True,
    # print_plots=True,
    # n_train_samples=180
)

/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/transforms/decomposition/detrend.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result[self.in_column] = no_trend_timeseries
/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/transforms/decomposition/detrend.py:96: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result[self.in_column] = no_trend_timeseries
/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etn

WAPE:0.5659079057890397


In [46]:
preprocess = [
    log, scaler, seg, lags, lags_2, d_flags,
    mean90, mean180, mean270, minmax270
]

forecast_ctbst = train_and_evaluate_model(
    ts=ts,
    model=ctbst_model,
    transforms=preprocess,
    horizon=HORIZON,
    metrics=wape,
    print_metrics=True,
    # print_plots=True,
    # n_train_samples=180
)

/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/datasets/utils.py:190: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long["segment"] = np.repeat(a=segments, repeats=n_timestamps)
/Users/sergey.rubtsovenko/Projects/research/.venv/lib/python3.12/site-packages/etna/datasets/utils.py:190: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_long["segment"] = np.repeat(a=segments, repeats=n_timestamps)


WAPE:0.5558425228100881
